# Phase 2 — Time-Series Baseline Forecasting

## Objective
Establish interpretable forecasting benchmarks before machine learning.

**Models:** Naive and 7-day Seasonal Naive.

**Evaluation:** train before 2024, validation in 2024, untouched test in 2025, with 7/30/90-day horizons and no random shuffling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = r"/mnt/data/smart_sales_forecasting_dataset/processed_daily_forecasting_features.csv"
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

daily = (
    df.groupby("Date", as_index=False)["Quantity"]
      .sum()
      .set_index("Date")
      .asfreq("D")
)

print("Shape:", daily.shape)
print("Date range:", daily.index.min().date(), "to", daily.index.max().date())
print("Missing target values:", daily["Quantity"].isna().sum())


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(daily.index, daily["Quantity"])
plt.title("Daily Aggregate Demand")
plt.xlabel("Date")
plt.ylabel("Quantity")
plt.tight_layout()
plt.show()


In [ ]:
TRAIN_END = pd.Timestamp("2023-12-31")
VAL_END = pd.Timestamp("2024-12-31")

train = daily.loc[:TRAIN_END]
validation = daily.loc["2024-01-01":"2024-12-31"]
test = daily.loc["2025-01-01":"2025-12-31"]

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))


In [ ]:
def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return np.sum(np.abs(y-p)) / np.sum(np.abs(y))

def evaluate(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    mask = y != 0
    return {
        "MAE": np.mean(np.abs(y-p)),
        "RMSE": np.sqrt(np.mean((y-p)**2)),
        "MAPE": np.mean(np.abs((y[mask]-p[mask])/y[mask])),
        "WAPE": wape(y,p)
    }

def naive(history, horizon):
    return np.repeat(float(history[-1]), horizon)

def seasonal_naive(history, horizon, season=7):
    history = list(np.asarray(history, float))
    preds = []
    for _ in range(horizon):
        pred = history[-season]
        preds.append(pred)
        history.append(pred)
    return np.asarray(preds)


In [ ]:
def rolling_baseline(test_series, history_series, horizon, method):
    history = list(history_series.astype(float).values)
    actual, predicted = [], []
    start = 0

    while start < len(test_series):
        h = min(horizon, len(test_series)-start)
        if method == "Naive":
            pred = naive(history, h)
        else:
            pred = seasonal_naive(history, h, 7)

        actual_block = test_series.iloc[start:start+h].values
        actual.extend(actual_block)
        predicted.extend(pred)

        # Move forecast origin forward using observations that have become known.
        history.extend(actual_block)
        start += h

    return np.asarray(actual), np.asarray(predicted)

rows, frames = [], []

for horizon in [7, 30, 90]:
    for model in ["Naive", "Seasonal Naive"]:
        y, p = rolling_baseline(test["Quantity"], train["Quantity"], horizon, model)
        rows.append({"Model": model, "Horizon_Days": horizon, **evaluate(y,p)})
        frames.append(pd.DataFrame({
            "Date": test.index,
            "Actual_Quantity": y,
            "Predicted_Quantity": p,
            "Model": model,
            "Horizon_Days": horizon
        }))

results = pd.DataFrame(rows).sort_values(["Horizon_Days","WAPE"])
predictions = pd.concat(frames, ignore_index=True)

display(results)


In [ ]:
best = results.sort_values(["Horizon_Days","WAPE"]).groupby(
    "Horizon_Days", as_index=False
).first()

display(best)

results.to_csv("/mnt/data/phase_2_baseline_results.csv", index=False)
best.to_csv("/mnt/data/phase_2_baseline_summary.csv", index=False)

plot = predictions[predictions["Horizon_Days"] == 30]
plt.figure(figsize=(14,5))
for model in ["Naive", "Seasonal Naive"]:
    x = plot[plot["Model"] == model]
    plt.plot(x["Date"], x["Predicted_Quantity"], label=model)

plt.plot(plot["Date"], plot["Actual_Quantity"], label="Actual")
plt.title("2025 Baseline Comparison — 30-Day Blocks")
plt.xlabel("Date")
plt.ylabel("Quantity")
plt.legend()
plt.tight_layout()
plt.show()


## Conclusion
The Seasonal Naive (7-day) model is the key benchmark because the demand series contains weekly seasonality.

**Scope:** aggregate daily `Quantity` forecasting. Product-level forecasting is a separate modeling problem.